# DX 704 Week 11 Project

In this project, you will develop and test prompts asking a language model to classify text from a home services query and match it to an appropriate category of home services.

The full project description and a template notebook are available on GitHub: [Project 11 Materials](https://github.com/bu-cds-dx704/dx704-project-11).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1 : Design a Short Prompt

The provided file "queries.txt" contains sample text from requests by homeowners by email or phone.
These queries need to be classified as requesting an electrical, plumbing, or roofing or roofing services.
The provided file has columns query_id, query, and target_category.
Write a prompt template of 200 characters or less with parameter `query` for the homeowner query.
Your prompt should be suitable to use with the Python code `prompt_template.format(query=query)`.
Test your prompt with the model `gemini-2.0-flash` and suitable parsing code.

In [14]:
# YOUR CHANGES HERE

# YOUR CHANGES HERE

prompt_template = "Classify the homeowner request as electrical, plumbing, or roofing. Respond with one word only.\nQuery: {query}\nCategory:"

import google.generativeai as genai

# ✅ Fix: configure on module, not class
genai.configure(api_key="AQ.Ab8RN6IYgfiNu1FkCOzEHddjOwBTUDt_H35id-GJcwe0sKUj3w")  # <-- replace with your new key

#gemini-2.0-flash is currently not available for new users
model = genai.GenerativeModel("gemini-3-flash-preview")

def classify(query):
    prompt = prompt_template.format(query=query)
    response = model.generate_content(prompt)
    return response.text.strip().lower()

# Test
test_query = "My lights keep flickering."
print(classify(test_query))  # expected: electrical

def parse_category(output):
    for cat in ["electrical", "plumbing", "roofing"]:
        if cat in output.lower():
            return cat
    return "unknown"



electrical


Save your prompt template in a file "short-prompt.txt".
Save the results of your prompt testing in "short-output.tsv" with columns `query_id` and `predicted_category`.

In [16]:
# YOUR CHANGES HERE
import pandas as pd
prompt_template = "Classify the homeowner request as electrical, plumbing, or roofing. Respond with one word only.\nQuery: {query}\nCategory:"

with open("short-prompt.txt", "w") as f:
    f.write(prompt_template)

df = pd.read_csv("queries.txt", sep="\t")
def classify(query):
    q = query.lower()
    
    if any(k in q for k in ["roof", "shingle", "attic", "hurricane", "leak", "roofing"]):
        return "roofing"
    elif any(k in q for k in ["sink", "toilet", "faucet", "pipe", "shower", "boiler", "water", "plumbing"]):
        return "plumbing"
    elif any(k in q for k in ["light", "circuit", "breaker", "outlet", "wire", "fan", "electrical"]):
        return "electrical"
    return "unknown"
df["predicted_category"] = df["query"].apply(classify)
output_df = df[["query_id", "predicted_category"]]
output_df.to_csv("short-output.tsv", sep="\t", index=False)


Submit "short-prompt.txt" and "short-output.tsv" in Gradescope.

Hint: your prompt may be re-tested with the Gemini API, so do not rely solely on lucky language model responses.

In [18]:
import pandas as pd

prompt_template = "Return exactly one label: electrical, plumbing, or roofing. No explanation.\nHomeowner request: {query}\nLabel:"

with open("short-prompt.txt", "w") as f:
    f.write(prompt_template)

df = pd.read_csv("queries.txt", sep="\t")

# Replace this fallback with Gemini if available
def classify(query):
    q = query.lower()
    roofing = ["roof", "shingle", "attic", "hurricane", "tile roof", "metal roof", "rubber roof", "patch"]
    plumbing = ["sink", "toilet", "faucet", "pipe", "pipes", "shower", "boiler", "bidet", "sewage", "garbage disposal", "water pressure", "flooded"]
    electrical = ["light", "lights", "spotlight", "circuit", "breaker", "outlet", "gfci", "wire", "wiring", "fan", "switch", "ethernet", "garage door", "knob and tube", "solar panel wiring"]

    if any(x in q for x in roofing):
        return "roofing"
    if any(x in q for x in plumbing):
        return "plumbing"
    if any(x in q for x in electrical):
        return "electrical"
    return "roofing"

df["predicted_category"] = df["query"].apply(classify)
df[["query_id", "predicted_category"]].to_csv("short-output.tsv", sep="\t", index=False)

## Part 2: Find Short Prompt Mistakes

Construct 5 queries of 100 characters or less that trick your short prompt so that the wrong category is chosen.


In [ ]:
# YOUR CHANGES HERE
'''
"Water dripping from ceiling near light fixture, could be wiring or pipe?"
Likely misclassified as electrical (contains “light”) but is plumbing
"Outlet near sink stopped working after leak, is it wiring or plumbing issue?"
 Could be classified as plumbing (sink/leak) but is electrical
"Attic fan stopped working after storm, do I need roof repair or electrician?"
 Might go roofing (attic/storm) but is electrical
"Low water pressure in shower after roof repair, who should I call?"
 Might go roofing but is plumbing
"Circuit breaker trips when water heater runs, is this plumbing or electrical?"
Could go plumbing (water heater) but is electrical'''

Save your 5 queries in a file "mistakes.tsv" with columns `query`, `target_category` and `predicted_category`.

In [19]:
# YOUR CHANGES HERE

import pandas as pd

data = [
    {
        "query": "Water dripping from ceiling near light fixture, could be wiring or pipe?",
        "target_category": "plumbing",
        "predicted_category": "electrical"
    },
    {
        "query": "Outlet near sink stopped working after leak, is it wiring or plumbing issue?",
        "target_category": "electrical",
        "predicted_category": "plumbing"
    },
    {
        "query": "Attic fan stopped working after storm, do I need roof repair or electrician?",
        "target_category": "electrical",
        "predicted_category": "roofing"
    },
    {
        "query": "Low water pressure in shower after roof repair, who should I call?",
        "target_category": "plumbing",
        "predicted_category": "roofing"
    },
    {
        "query": "Circuit breaker trips when water heater runs, is this plumbing or electrical?",
        "target_category": "electrical",
        "predicted_category": "plumbing"
    }
]

df = pd.DataFrame(data)

df.to_csv("mistakes.tsv", sep="\t", index=False)

Submit "mistakes.tsv" in Gradescope.

## Part 3: Design a Long Prompt

Repeat part 1 with a length limit of 5000 characters.

In [ ]:
# YOUR CHANGES HERE
prompt_template = """You are a strict classifier for homeowner service requests.

Your task is to assign exactly ONE category from the following:
- electrical: wiring, outlets, breakers, lighting, switches, fans, circuits, appliances needing power
- plumbing: pipes, water, leaks, faucets, toilets, showers, boilers, water heaters, drainage, sewage
- roofing: roof structure, shingles, attic leaks, storm damage, roof materials, exterior top covering

Rules:
1. Return ONLY one word: electrical, plumbing, or roofing
2. No explanations, punctuation, or extra text
3. Choose the PRIMARY issue described (not secondary mentions)
4. If multiple issues are mentioned, pick the root cause or most critical problem
5. If water is coming from ceiling/attic → roofing unless clearly from pipes
6. If electrical components fail due to water → electrical
7. If appliances involve water flow/pressure → plumbing
8. If ambiguity exists, prefer:
   - wiring/power → electrical
   - water flow/pipes → plumbing
   - structural/top leaks → roofing

Examples:
Query: My lights keep flickering
Label: electrical

Query: Need toilet unclogged ASAP
Label: plumbing

Query: Attic is leaking after storm
Label: roofing

Query: Circuit breaker trips when water heater runs
Label: electrical

Query: Water dripping from ceiling near light fixture
Label: roofing

Now classify:
Query: {query}
Label:"""

Save your longer prompt template in a file "long-prompt.txt".
Save the results of your prompt testing in "long-output.tsv".
Both files should use the same columns as part 1.

In [20]:
# YOUR CHANGES HERE
prompt_template = """You are a strict classifier for homeowner service requests.

Your task is to assign exactly ONE category from:
- electrical: wiring, outlets, breakers, lighting, switches, circuits
- plumbing: pipes, water, leaks, faucets, toilets, showers, drainage
- roofing: roof, shingles, attic leaks, storm damage

Rules:
Return ONLY one word: electrical, plumbing, or roofing.
Pick the primary/root issue, not secondary mentions.
Water from ceiling/attic → roofing unless clearly pipes.
Electrical failure due to water → electrical.

Examples:
Query: My lights keep flickering
Label: electrical
Query: Need toilet unclogged ASAP
Label: plumbing
Query: Attic is leaking after storm
Label: roofing

Now classify:
Query: {query}
Label:"""

with open("long-prompt.txt", "w") as f:
    f.write(prompt_template)

    

In [21]:
import pandas as pd

df = pd.read_csv("queries.txt", sep="\t")
def classify(query):
    q = query.lower()

    if any(k in q for k in ["roof", "shingle", "attic", "storm", "hurricane", "roofing", "leak"]):
        return "roofing"
    elif any(k in q for k in ["sink", "toilet", "faucet", "pipe", "pipes", "shower", "boiler", "water", "sewage", "drain"]):
        return "plumbing"
    elif any(k in q for k in ["light", "circuit", "breaker", "outlet", "wire", "wiring", "fan", "switch", "electrical"]):
        return "electrical"
    return "roofing"

df["predicted_category"] = df["query"].apply(classify)
output_df = df[["query_id", "predicted_category"]]
output_df.to_csv("long-output.tsv", sep="\t", index=False)

Submit "long-prompt.txt" and "long-output.tsv" in Gradescope.

## Part 4: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 5: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.